In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import mplfinance as mpf # k线图
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体显示
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']  # macos
# plt.rcParams['font.sans-serif'] = ['SimHei']  # windows
plt.rcParams['axes.unicode_minus'] = False

### 创建你的第一个量化策略
1. 参数设置：ticker 数据获取与标准化
2. 参数设置：captical, commission, benchmark
3. 策略与运行：indicators, signals
4. 策略与运行：cross-order, limit, pos, 涨跌停
5. 统计与验证：daily result (pnl)
6. 统计与验证：strategy statistics
7. 结果可视化：持仓收益、最大回撤、日盈亏
8. 结果可视化：市场基准（beta）对比

In [6]:
# 全局参数设置：工商银行为例
ticker = "601398.SS"
start_date = "2024-12-31"
end_date = "2025-03-01"

#### 参数设置：ticker 数据获取与标准化
- 数据获取：yf.download(), 定义参数（股票代码、开始日期、结束日期、复权方式）
- 数据清洗：nan值 -> dropna
- 数据预览：整体看下数据

In [6]:
# 函数1：数据获取
def fetcher_data(ticker:str,start_date:str,end_date:str,auto_clean=True) -> pd.DataFrame:
    """ 获取数据 """
    try:
        print("正在获取历史数据：",ticker, start_date, end_date)
        data = yf.download(tickers=ticker,start=start_date,end=end_date,progress=False)
        print("完成数据获取，共有记录（条数）：", len(data))

        if auto_clean:
            clean_data(data)
    
        return data
    except Exception as e:
        print("获取数据异常：",e)
        return pd.DataFrame

# 函数2：数据清洗
def clean_data(data:pd.DataFrame) -> pd.DataFrame:
    """ 清洗数据：缺失值 """
    try:
        count_nan = data.isnull().sum().sum()
        print("当前data共有缺失值（个数）：",count_nan)
        data = data.dropna()
        count_nan = data.isnull().sum().sum()
        print("缺失值处理完毕，当前缺失值（个数）：",count_nan)
        return data
    except Exception as e:
        print("获取数据异常：",e)
        return data

# 函数3：数据预览
def preview_data(data:pd.DataFrame) -> pd.DataFrame:
    """ 按需要的格式预览数据 """
    print("数据预览：",data)

In [10]:
data = fetcher_data(ticker,start_date,end_date)
preview_data(data)

正在获取历史数据： 601398.SS 2024-12-31 2025-03-01
完成数据获取，共有记录（条数）： 37
当前data共有缺失值（个数）： 0
缺失值处理完毕，当前缺失值（个数）： 0
数据预览： Price          Close      High       Low      Open     Volume
Ticker     601398.SS 601398.SS 601398.SS 601398.SS  601398.SS
Date                                                         
2024-12-31  6.633808  6.739259  6.614635  6.652981  520980174
2025-01-02  6.518771  6.691327  6.480425  6.624222  531482465
2025-01-03  6.432493  6.547530  6.346215  6.528357  521073986
2025-01-06  6.470839  6.499599  6.317456  6.432493  474224652
2025-01-07  6.552525  6.572114  6.425196  6.474169  382021777
2025-01-08  6.621087  6.699443  6.532936  6.552525  399684284
2025-01-09  6.572114  6.630881  6.523142  6.621087  236723419
2025-01-10  6.532936  6.601497  6.464374  6.581908  262025788
2025-01-13  6.464375  6.532936  6.356635  6.523142  330603282
2025-01-14  6.474169  6.513347  6.425197  6.454580  341885264
2025-01-15  6.493758  6.591703  6.444785  6.464374  307598274
2025-01-16  6.542730  6.